# Predict FPL Points
This notebook predicts FPL points using the data ingested and the model trained in previous notebooks.

## 1. Include required libraries

In [15]:
import joblib
import pandas as pd

## 2. Load Trained Model
We load the model that was trained in a previous notebook

In [16]:
# Load the trained model
model_filename = "../data/model/random_forest/20250406_162635/model.joblib"
model = joblib.load(model_filename)

## 3. Load Data
We load the features that were extracted and transformed from a previous notebook

In [17]:
# Load data
data_filename = "../data/processed/features/20250406_162326/features.parquet"
data = pd.read_parquet(data_filename)

# Show the data
data.head()

,fixture_code,team_code,opponent_team_code,kickoff_time,season_code,player_name,player_code,team_name,opponent_team_name,position,...,recent_team_clean_sheets,long_term_team_clean_sheets,recent_team_total_points,long_term_team_total_points,position_GK,position_DEF,position_MID,position_FWD,was_home_true,was_home_false
0,2292813,14,54,2022-08-06 11:30:00,2022_23,James Milner,15157,Liverpool,Fulham,MID,...,NaN,NaN,NaN,NaN,0,0,1,0,0,1
1,2292825,14,31,2022-08-15 19:00:00,2022_23,James Milner,15157,Liverpool,Crystal Palace,MID,...,NaN,NaN,NaN,NaN,0,0,1,0,1,0
2,2292836,14,1,2022-08-22 19:00:00,2022_23,James Milner,15157,Liverpool,Man Utd,MID,...,NaN,NaN,NaN,NaN,0,0,1,0,0,1
3,2292845,14,91,2022-08-27 14:00:00,2022_23,James Milner,15157,Liverpool,Bournemouth,MID,...,NaN,NaN,NaN,NaN,0,0,1,0,1,0
4,2292857,14,4,2022-08-31 19:00:00,2022_23,James Milner,15157,Liverpool,Newcastle,MID,...,1.0,NaN,214.0,NaN,0,0,1,0,1,0


In [18]:
impute_cols = [
    'recent_goals_scored',
    'long_term_goals_scored',
    'recent_goals_conceded',
    'long_term_goals_conceded',
    'recent_assists',
    'long_term_assists',
    'recent_expected_assists',
    'long_term_expected_assists',
    'recent_expected_goal_involvements',
    'long_term_expected_goal_involvements',
    'recent_expected_goals',
    'long_term_expected_goals',
    'recent_expected_goals_conceded',
    'long_term_expected_goals_conceded',
    'recent_penalties_missed',
    'long_term_penalties_missed',
    'recent_penalties_saved',
    'long_term_penalties_saved',
    'recent_saves',
    'long_term_saves',
    'recent_bps',
    'long_term_bps',
    'recent_minutes',
    'long_term_minutes',
    'recent_yellow_cards',
    'long_term_yellow_cards',
    'recent_red_cards',
    'long_term_red_cards',
    'recent_own_goals',
    'long_term_own_goals',
    'recent_starts',
    'long_term_starts',
    'recent_influence',
    'long_term_influence',
    'recent_creativity',
    'long_term_creativity',
    'recent_threat',
    'long_term_threat',
    'recent_ict_index',
    'long_term_ict_index',
    'recent_clean_sheets',
    'long_term_clean_sheets',
    'recent_total_points',
    'long_term_total_points',
    'recent_team_goals_scored',
    'long_term_team_goals_scored',
    'recent_team_goals_conceded',
    'long_term_team_goals_conceded',
    'recent_team_assists',
    'long_term_team_assists',
    'recent_team_expected_assists',
    'long_term_team_expected_assists',
    'recent_team_expected_goal_involvements',
    'long_term_team_expected_goal_involvements',
    'recent_team_expected_goals',
    'long_term_team_expected_goals',
    'recent_team_expected_goals_conceded',
    'long_term_team_expected_goals_conceded',
    'recent_team_penalties_missed',
    'long_term_team_penalties_missed',
    'recent_team_penalties_saved',
    'long_term_team_penalties_saved',
    'recent_team_saves',
    'long_term_team_saves',
    'recent_team_bps',
    'long_term_team_bps',
    'recent_team_yellow_cards',
    'long_term_team_yellow_cards',
    'recent_team_red_cards',
    'long_term_team_red_cards',
    'recent_team_own_goals',
    'long_term_team_own_goals',
    'recent_team_clean_sheets',
    'long_term_team_clean_sheets',
    'recent_team_total_points',
    'long_term_team_total_points',
    'position_GK',
    'position_DEF',
    'position_MID',
    'position_FWD'
]

In [19]:
# Impute future values using the most recent available data
def impute_most_recent(df, group_cols, value_cols, date_col):
    """
    Imputes missing values in a Pandas DataFrame using the most recent available data
    within specified groups.

    Args:
        df (pd.DataFrame): The DataFrame to impute.
        group_cols (list): List of columns to group by.
        value_col (str): The column containing the values to impute.
        date_col (str): The column containing the dates.

    Returns:
        pd.DataFrame: The DataFrame with imputed values.
    """

    # Sort values by group columns and date column
    df = df.sort_values(by=[*group_cols, date_col])

    # Forward fill the values within each group
    for col in value_cols:
        df[col] = df.groupby(group_cols)[col].ffill()

    return df

# Impute future player features
group_cols = ["player_code"]
value_cols = impute_cols
sort_by_col = "kickoff_time"
imputed_data = impute_most_recent(data, group_cols, value_cols, sort_by_col)

In [20]:
# Validate imputed data
imputed_data[(imputed_data['season_code'] == '2024_25') & (imputed_data['player_code'] == 15157)].sort_values("gameweek", ascending=False).head(10)

,fixture_code,team_code,opponent_team_code,kickoff_time,season_code,player_name,player_code,team_name,opponent_team_name,position,...,recent_team_clean_sheets,long_term_team_clean_sheets,recent_team_total_points,long_term_team_total_points,position_GK,position_DEF,position_MID,position_FWD,was_home_true,was_home_false
151,2444848,36,6,2025-05-25 15:00:00,2024_25,James Milner,15157,Brighton,Spurs,MID,...,0.0,2.0,139.0,389.0,0,0,1,0,0,1
150,2444833,36,14,2025-05-18 14:00:00,2024_25,James Milner,15157,Brighton,Liverpool,MID,...,0.0,2.0,139.0,389.0,0,0,1,0,1,0
149,2444829,36,39,2025-05-10 14:00:00,2024_25,James Milner,15157,Brighton,Wolves,MID,...,0.0,2.0,139.0,389.0,0,0,1,0,0,1
148,2444813,36,4,2025-05-04 13:00:00,2024_25,James Milner,15157,Brighton,Newcastle,MID,...,0.0,2.0,139.0,380.0,0,0,1,0,1,0
147,2444802,36,21,2025-04-26 14:00:00,2024_25,James Milner,15157,Brighton,West Ham,MID,...,0.0,2.0,139.0,409.0,0,0,1,0,1,0
146,2444791,36,94,2025-04-19 14:00:00,2024_25,James Milner,15157,Brighton,Brentford,MID,...,0.0,3.0,139.0,444.0,0,0,1,0,0,1
145,2444782,36,13,2025-04-12 14:00:00,2024_25,James Milner,15157,Brighton,Leicester,MID,...,0.0,3.0,139.0,421.0,0,0,1,0,1,0
144,2444772,36,31,2025-04-05 14:00:00,2024_25,James Milner,15157,Brighton,Crystal Palace,MID,...,0.0,3.0,157.0,407.0,0,0,1,0,0,1
143,2444762,36,7,2025-04-02 18:45:00,2024_25,James Milner,15157,Brighton,Aston Villa,MID,...,1.0,3.0,231.0,372.0,0,0,1,0,1,0
142,2444757,36,43,2025-03-15 15:00:00,2024_25,James Milner,15157,Brighton,Man City,MID,...,2.0,2.0,298.0,306.0,0,0,1,0,0,1


## 4. Predict FPL Points
We predict FPL points from the new data.

In [24]:
predicted_data = imputed_data

# Drop columns from data that are not in the features that the model was trained on
drop_cols = [col for col in predicted_data.columns if col not in model.feature_names_in_]
features = imputed_data.drop(columns=drop_cols)

# Make predictions
predictions = model.predict(features)

# Add predictions to the original data
predicted_data["predicted_points"] = predictions

# Show the data with predictions
predicted_data.head()

,fixture_code,team_code,opponent_team_code,kickoff_time,season_code,player_name,player_code,team_name,opponent_team_name,position,...,long_term_team_clean_sheets,recent_team_total_points,long_term_team_total_points,position_GK,position_DEF,position_MID,position_FWD,was_home_true,was_home_false,predicted_points
15069,2293124,11,31,2023-04-22 14:00:00,2022_23,Andy Lonergan,11948,Everton,Crystal Palace,GK,...,3.0,120.0,282.0,1,0,0,0,0,1,0.0
15070,2293131,11,4,2023-04-27 18:45:00,2022_23,Andy Lonergan,11948,Everton,Newcastle,GK,...,3.0,131.0,303.0,1,0,0,0,1,0,0.0
15071,2293146,11,13,2023-05-01 19:00:00,2022_23,Andy Lonergan,11948,Everton,Leicester,GK,...,2.0,118.0,277.0,1,0,0,0,0,1,0.0
15072,2293151,11,36,2023-05-08 16:30:00,2022_23,Andy Lonergan,11948,Everton,Brighton,GK,...,2.0,132.0,282.0,1,0,0,0,0,1,0.0
15073,2293165,11,43,2023-05-14 13:00:00,2022_23,Andy Lonergan,11948,Everton,Man City,GK,...,1.0,164.0,248.0,1,0,0,0,1,0,0.0


## 5. Clean Data
We clean the data and select which columns we want to export for visualisation.

In [27]:
# Create a DataFrame with the predictions and the target variable
cleaned_data = predicted_data

cleaned_data.head()

,fixture_code,team_code,opponent_team_code,kickoff_time,season_code,player_name,player_code,team_name,opponent_team_name,position,...,long_term_team_clean_sheets,recent_team_total_points,long_term_team_total_points,position_GK,position_DEF,position_MID,position_FWD,was_home_true,was_home_false,predicted_points
15069,2293124,11,31,2023-04-22 14:00:00,2022_23,Andy Lonergan,11948,Everton,Crystal Palace,GK,...,3.0,120.0,282.0,1,0,0,0,0,1,0.0
15070,2293131,11,4,2023-04-27 18:45:00,2022_23,Andy Lonergan,11948,Everton,Newcastle,GK,...,3.0,131.0,303.0,1,0,0,0,1,0,0.0
15071,2293146,11,13,2023-05-01 19:00:00,2022_23,Andy Lonergan,11948,Everton,Leicester,GK,...,2.0,118.0,277.0,1,0,0,0,0,1,0.0
15072,2293151,11,36,2023-05-08 16:30:00,2022_23,Andy Lonergan,11948,Everton,Brighton,GK,...,2.0,132.0,282.0,1,0,0,0,0,1,0.0
15073,2293165,11,43,2023-05-14 13:00:00,2022_23,Andy Lonergan,11948,Everton,Man City,GK,...,1.0,164.0,248.0,1,0,0,0,1,0,0.0


## 6. Write Data

In [28]:
from datetime import datetime
import os

# Define output directory
# We define the directory where the processed data will be saved.
data_source = "predictions"
current_datetime = datetime.now().strftime("%Y%m%d_%H%M%S")
output_dir = f"../data/processed/{data_source}/{current_datetime}"
filename = "predictions.parquet"
filepath = os.path.join(output_dir, filename)

if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# Create the directory if it doesn't exist
# We create the directory if it doesn't exist.
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# # Write the transformed DataFrames to Parquet files
# # We save the processed data as parquet files.
cleaned_data.to_parquet(filepath, index=False)

print(f"Predictions complete! Data saved to {output_dir}")

Predictions complete! Data saved to ../data/processed/predictions/20250406_163428
